# LlmEdith : PyTorch sur 2×T4 (Kaggle)
*Accelerator* = **GPU T4 x2**, *Internet* = **ON**, secrets `HF_TOKEN` / `GITHUB_TOKEN`.
Remplace un GPU local : ablations, **benchmarks** des checkpoints de la run TPU, puis **SFT/DPO**.
Les T4 n'ont pas de bf16 : l'entraînement passe automatiquement en fp16 + GradScaler.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
try:
    GH = s.get_secret('GITHUB_TOKEN')  # seulement si le dépôt GitHub est privé
except Exception:
    GH = None
os.environ['LLMEDITH_HUB_REPO'] = 'HarpePluie/llmedith-checkpoints'
repo = 'https://github.com/HarpeLm/LlmEdith.git'.replace('https://', f'https://{GH}@') if GH else 'https://github.com/HarpeLm/LlmEdith.git'
if not os.path.exists('LlmEdith'):
    subprocess.run(['git', 'clone', '-q', repo, 'LlmEdith'], check=True)
else:
    subprocess.run(['git', '-C', 'LlmEdith', 'pull', '-q'], check=True)
os.chdir('LlmEdith')

In [ ]:
!pip install -q -e '.[eval]'

## A. Ablation `tiny` : Muon contre AdamW, un modèle par GPU, en parallèle
Environ 3 h. Relance la cellule dans une nouvelle session si elle est coupée : chaque run reprend.

In [ ]:
import subprocess
def run(gpu, name, opt):
    cmd = (f"CUDA_VISIBLE_DEVICES={gpu} python -m llmedith.torch_impl.train --config configs/tiny.yaml "
           f"--resume --max_minutes 480 --override name={name} train.optimizer={opt} data.hub_repo=HarpePluie/llmedith-data")
    return subprocess.Popen(cmd + f" > {name}.log 2>&1", shell=True)
procs = [run(0, 'tiny_muon', 'muon'), run(1, 'tiny_adamw', 'adamw')]
for p in procs:
    p.wait()
!tail -n 3 tiny_muon.log tiny_adamw.log

In [ ]:
import pandas as pd
cols = ['step', 'split', 'loss', 'lr', 'gnorm', 'tok_s']
ax = None
for name in ['tiny_muon', 'tiny_adamw']:
    log = pd.read_csv(f'runs/{name}/log.csv', names=cols)
    val = log[log.split == 'val']
    ax = val.plot(x='step', y='loss', ax=ax, label=name, style='o-')
    print(name, 'val loss finale :', val.loss.iloc[-1])

## B. Benchmarks du dernier checkpoint de la run principale (TPU)

In [ ]:
from llmedith import hub
hub.pull_folder('HarpePluie/llmedith-checkpoints', 'main/latest', 'runs')
!python -m convert.to_hf --ckpt runs/main/latest --out exports/llmedith-main
!python -m eval.run_benchmarks --models edith-main=exports/llmedith-main --baselines --force

## C. Alignement : SFT puis DPO (quand le pré-entraînement est terminé)
Chaque étape est envoyée sur le Hub à la fin, pour pouvoir continuer dans une autre session.

In [ ]:
from llmedith import hub
hub.pull_folder('HarpePluie/llmedith-checkpoints', 'main/latest', 'runs')
!python -m posttrain.sft --base runs/main/latest --out runs/sft --max_examples 100000 --epochs 2
hub.push_folder('HarpePluie/llmedith-checkpoints', 'runs/sft/latest', 'sft/latest', blocking=True)

In [ ]:
from llmedith import hub
if not os.path.exists('runs/sft/latest/meta.json'):
    hub.pull_folder('HarpePluie/llmedith-checkpoints', 'sft/latest', 'runs')
!python -m posttrain.dpo --base runs/sft/latest --out runs/dpo --max_examples 20000
hub.push_folder('HarpePluie/llmedith-checkpoints', 'runs/dpo/latest', 'dpo/latest', blocking=True)